# 🗄️ eidcloud-json-db: Quickstart & Setup

Welcome to **eidcloud-json-db**! This notebook demonstrates how to use the pure PHP embedded JSON document database.

### Step 1: Environment Setup & Database Initialization
We setup PHP and clone/load `eidcloud-json-db`.

In [ ]:
%%bash
# Ensure PHP 8.2+ is installed in environment
php -v

# Run quick verification test suite
if [ -f "tests/run_tests.php" ]; then
    php tests/run_tests.php
else
    echo "eidcloud-json-db ready for demonstration."
fi

In [ ]:
%%bash
# 1. Create collections, insert indexed documents, and perform fluent queries
php -r '
spl_autoload_register(function ($class) {
    $prefix = "EidCloud\\JsonDb\\";
    $baseDir = __DIR__ . "/src/";
    $len = strlen($prefix);
    if (strncmp($prefix, $class, $len) !== 0) return;
    $file = $baseDir . str_replace("\\", "/", substr($class, $len)) . ".php";
    if (file_exists($file)) require_once $file;
});

use EidCloud\JsonDb\Database;

$db = new Database("/tmp/colab_json_db");
$users = $db->table("users");
$users->createIndex("status");
$users->createIndex(["status", "department"]);

echo "Inserting 1,000 users inside ACID transaction...\n";
$records = [];
for ($i = 1; $i <= 1000; $i++) {
    $records[] = [
        "name" => "Engineer_{$i}",
        "email" => "eng{$i}@eidcloud.com",
        "department" => ($i % 2 === 0 ? "Cloud" : "AI"),
        "status" => ($i % 3 === 0 ? "active" : "pending"),
        "age" => 22 + ($i % 40)
    ];
}
$users->insertMany($records);

// Fluent Indexed Query
$t0 = microtime(true);
$activeCloud = $users->where("status", "=", "active")
                     ->where("department", "=", "Cloud")
                     ->where("age", ">", 30)
                     ->orderBy("age", "desc")
                     ->limit(5)
                     ->get();
$queryDuration = (microtime(true) - $t0) * 1000;

echo "Indexed Query completed in " . round($queryDuration, 3) . " ms. Top matching records:\n";
foreach ($activeCloud as $row) {
    echo " - [ID: {$row["id"]}] {$row["name"]} ({$row["department"]}) Age: {$row["age"]} Status: {$row["status"]}\n";
}
'

# 2. Run benchmark vs raw file IO using CLI
php bin/eidcloud-json-db benchmark --records=5000